In [ ]:
#@title Import
import os
# os.environ["JAX_PLATFORMS"] = 'cpu'

import math
import torch

import sys 
sys.path.append('..')

import jax
jax.config.update("jax_enable_x64", True)

from jax import numpy as jnp
from jax import random as jr
from jax import vmap, lax
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import numpy as np

from dynamax.utils.plotting import plot_uncertainty_ellipses
from dynamax.linear_gaussian_ssm import LinearGaussianSSM, LinearGaussianConjugateSSM
from dynamax.linear_gaussian_ssm.inference import lgssm_filter, lgssm_smoother, lgssm_posterior_sample
from dynamax.linear_gaussian_ssm.inference import ParamsLGSSM, ParamsLGSSMInitial, ParamsLGSSMDynamics, ParamsLGSSMEmissions
from dynamax.linear_gaussian_ssm.inference import PosteriorGSSMFiltered, PosteriorGSSMSmoothed
from dynamax.parameters import ParameterProperties, ParameterSet
from dynamax.utils.utils import monotonically_increasing, random_rotation, pytree_slice, rotate_subspace, random_dynamics_weights, gram_schmidt
from dynamax.utils.distributions import NormalInverseWishart as NIW
from dynamax.nonlinear_gaussian_ssm import StiefelManifoldSSM
from dynamax.linear_gaussian_ssm.inference import ParamsLGSSMInitial, ParamsLGSSMDynamics
from dynamax.nonlinear_gaussian_ssm.models import ParamsSMDS, ParamsSMDSEmissions
from tensorflow_probability.substrates.jax.distributions import MultivariateNormalFullCovariance as MVN
from tensorflow_probability.substrates.jax.distributions import InverseGamma as IG
from dynamax.linear_gaussian_ssm.inference import make_lgssm_params


from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import r2_score, explained_variance_score
from sklearn.cluster import KMeans

from scipy.spatial import distance_matrix, distance
from scipy.stats import chi2
import scipy.signal as signal
from scipy.special import logsumexp
from scipy.ndimage import gaussian_filter, gaussian_filter1d

from sklearn.decomposition import PCA

from tensorflow_probability.substrates import jax as tfp

import torch
import torch.nn.functional as F

from tqdm import tqdm

import h5py
import pandas as pd
import csv
import pickle as pkl

from collections import defaultdict

import time

import wandb
import re
from dynamax.linear_gaussian_ssm.models import ConditionallyLinearGaussianSSM
from dynamax.utils.utils import Tm_basis

os.makedirs('figure2_clds', exist_ok=True)
os.makedirs('appendix_clds', exist_ok=True)


In [ ]:
def polar_SO(M: torch.Tensor):
    """
    Closest rotation (SO(D)) to M via SVD; avoids reflections.
    Args:
        M: (..., D, D) real tensor
    Returns:
        R: (..., D, D) rotation (det=+1)
        s: (..., D) singular values of M
    """
    U, s, Vh = torch.linalg.svd(M, full_matrices=False)          # M ≈ U @ diag(s) @ Vh
    det = torch.det(U @ Vh)                                      # (...,)

    D = U.shape[-1]
    J = torch.eye(D, dtype=M.dtype, device=M.device)             # (D, D)
    J = J.expand(U.shape[:-2] + (D, D)).clone()                  # (..., D, D)
    J[..., -1, -1] = torch.where(det < 0, M.new_tensor(-1.0), M.new_tensor(1.0))

    R = U @ J @ Vh
    return R, s


# def within_twist(R: torch.Tensor):
#     """
#     Exact within-plane twist from eigen-phases of R ∈ O(D).
#     Returns sqrt(sum(theta_i^2)) where eigenvalues are e^{± i theta_i}.
#     Works on CPU or GPU (torch.linalg.eigvals is CUDA-capable).
#     """
#     evals = torch.linalg.eigvals(R)                              # (..., D) complex
#     # print(evals.shape)
#     angles = torch.abs(torch.angle(evals))                       # (..., D) in [0, π]
#     # return torch.linalg.norm(angles, dim=-1) / math.sqrt(2.0)    # (...,)
#     return torch.linalg.norm(angles[:, :, ::2], dim=-1)    # (...,)

def within_twist(R: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    """
    Geodesic distance on SO(D) from eigen-phases.
    Returns sqrt(sum(theta_i^2)) where eigenvalues are e^{± i theta_i}.
    """
    evals = torch.linalg.eigvals(R)                  # (..., D) complex
    angles = torch.abs(torch.angle(evals))           # (..., D) in [0, π]
    # kill tiny numerical noise near 0
    angles = torch.where(angles < eps, angles.new_zeros(()), angles)
    return torch.linalg.norm(angles, dim=-1) / math.sqrt(2.0)

def twist_angles(A: torch.Tensor, B: torch.Tensor):
    """
    Scalar geodesic 'twist' between A and B (rotation-only, ignores scale/shear).
    A: (..., N, D) with orthonormal columns (A^T A = I)
    B: (..., N, D) with orthonormal columns (B^T B = I)
    """
    M = A.transpose(-1, -2) @ B          # (..., D, D)
    R, _ = polar_SO(M)
    return within_twist(R)               # (...,)


# ---------- Pairwise T×T twist for weights ∈ R^{T×N×D} ----------

def compute_twist_angle_matrix(weights: torch.Tensor):
    """
    weights: (T, N, D), each slice weights[t] has orthonormal columns (Stiefel: N×D).
    Returns:
        twists: (T, T) where twists[i, j] is the rotation-only geodesic twist
                between weights[i] and weights[j].
    """
    assert weights.ndim == 3, "weights must be (T, N, D)"
    T, N, D = weights.shape

    # Build all pairwise M_ij = A_i^T @ B_j  →  shape (T, T, D, D)
    A_T = weights.transpose(-1, -2)                  # (T, D, N)
    M = A_T[:, None, ...] @ weights[None, :, ...]    # (T, 1, D, N) @ (1, T, N, D) -> (T, T, D, D)

    # Project to closest rotations and compute twists
    R, _ = polar_SO(M)                                # (T, T, D, D)
    twists = within_twist(R)                          # (T, T)
    return twists




# def polar_SO(M: torch.Tensor):
#     """
#     Closest rotation (SO(D)) to M via SVD; avoids reflections.
#     Args:
#         M: (..., D, D) real tensor
#     Returns:
#         R: (..., D, D) rotation (det=+1)
#         s: (..., D) singular values of M
#     """
#     U, s, Vh = torch.linalg.svd(M, full_matrices=False)          # M ≈ U @ diag(s) @ Vh
#     det = torch.det(U @ Vh)                                      # (...,)

#     D = U.shape[-1]
#     J = torch.eye(D, dtype=M.dtype, device=M.device)             # (D, D)
#     J = J.expand(U.shape[:-2] + (D, D)).clone()                  # (..., D, D)
#     J[..., -1, -1] = torch.where(det < 0, M.new_tensor(-1.0), M.new_tensor(1.0))

#     R = U @ J @ Vh
#     return R, s


# def within_twist(R: torch.Tensor):
#     """
#     Exact within-plane twist from eigen-phases of R ∈ O(D).
#     Returns sqrt(sum(theta_i^2)) where eigenvalues are e^{± i theta_i}.
#     Works on CPU or GPU (torch.linalg.eigvals is CUDA-capable).
#     """
#     evals = torch.linalg.eigvals(R)                              # (..., D) complex
#     # print(evals.shape)
#     angles = torch.abs(torch.angle(evals))                       # (..., D) in [0, π]
#     return torch.linalg.norm(angles, dim=-1) / math.sqrt(2.0)    # (...,)

# def twist_angles(A: torch.Tensor, B: torch.Tensor):
#     """
#     Scalar geodesic 'twist' between A and B (rotation-only, ignores scale/shear).
#     A: (..., N, D) with orthonormal columns (A^T A = I)
#     B: (..., N, D) with orthonormal columns (B^T B = I)
#     """
#     M = A.transpose(-1, -2) @ B          # (..., D, D)
#     R, _ = polar_SO(M)
#     return within_twist(R)               # (...,)


# # ---------- Pairwise T×T twist for weights ∈ R^{T×N×D} ----------

# def compute_twist_angle_matrix(weights: torch.Tensor):
#     """
#     weights: (T, N, D), each slice weights[t] has orthonormal columns (Stiefel: N×D).
#     Returns:
#         twists: (T, T) where twists[i, j] is the rotation-only geodesic twist
#                 between weights[i] and weights[j].
#     """
#     assert weights.ndim == 3, "weights must be (T, N, D)"
#     T, N, D = weights.shape

#     # Build all pairwise M_ij = A_i^T @ B_j  →  shape (T, T, D, D)
#     A_T = weights.transpose(-1, -2)                  # (T, D, N)
#     M = A_T[:, None, ...] @ weights[None, :, ...]    # (T, 1, D, N) @ (1, T, N, D) -> (T, T, D, D)

#     # Project to closest rotations and compute twists
#     R, _ = polar_SO(M)                                # (T, T, D, D)
#     twists = within_twist(R)                          # (T, T)
#     return twists




def subspace_angles(A, B):
    """
    Compute the angle between two subspaces
    https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.subspace_angles.html
    """

    # 1. Compute orthonormal bases of column spaces
    QA = jnp.linalg.svd(A, full_matrices=False)[0]
    QB = jnp.linalg.svd(B, full_matrices=False)[0]

    # 2. Compute SVD for cosine
    QA_H_QB = jnp.dot(QA.T.conj(), QB)
    sigma = jnp.linalg.svd(QA_H_QB, full_matrices=False, compute_uv=False)

    # 3. Compute matrix B
    B = QB - jnp.dot(QA, QA_H_QB)

    # 4. Compute SVD for sine
    mask = sigma ** 2 >= 0.5
    mu_arcsin = jnp.arcsin(jnp.clip(jnp.linalg.svd(B, full_matrices=False, compute_uv=False), a_min=-1., a_max=1.))

    # 5. Compute the principal angles
    theta = jnp.where(mask, mu_arcsin, jnp.arccos(jnp.clip(sigma[::-1], a_min=-1., a_max=1.)))

    return theta

def compute_subspace_angle_matrix_v2(weights):
    # weights: T x N x D
    # output: T x T

    T = weights.shape[0]

    def _compute_subspace_angles(carry, anchor):
        sa = vmap(subspace_angles, in_axes=(0, None))(weights, weights[anchor])
        return None, sa

    _, subspace_angle_matrix = lax.scan(_compute_subspace_angles, init=None, xs=jnp.arange(T))

    return subspace_angle_matrix

def compute_rotation(observations, emissions):
    """
    Given:
      observations: jnp.ndarray of shape (num_trials, num_timesteps, emission_dim)
      emissions:    jnp.ndarray of shape (num_trials, emission_dim, latent_dim)
    Returns:
      R: jnp.ndarray of shape (latent_dim, latent_dim)
         A rotation matrix such that rotating each trial's emission matrix (i.e. computing O_i @ R)
         orders the latent columns in descending order of explained variance.
      explained_variances: jnp.ndarray of shape (latent_dim,)
         Variances explained by the rotated latent dimensions.
    """
    # Compute latent factors per trial: shape (num_trials, num_timesteps, latent_dim)
    # Using the fact that the emission matrices are orthogonal.
    latent_factors = jnp.einsum('...te,...el->...tl', observations, emissions)
    
    # Stack trials and timesteps together, so we have all latent factors in one 2D array.
    if latent_factors.ndim == 3:
        num_trials, num_timesteps, latent_dim = latent_factors.shape
    else:
        num_timesteps, latent_dim = latent_factors.shape
        num_trials = 1

    latent_factors_stacked = latent_factors.reshape(num_trials * num_timesteps, latent_dim)
    
    # Perform SVD on the aggregated latent factors.
    # latent_factors_stacked = U @ diag(S) @ Vt, with singular values S in descending order.
    U, S, Vt = jnp.linalg.svd(latent_factors_stacked, full_matrices=False)
    
    # The rotation matrix that aligns with the principal directions is given by V = Vt.T.
    # Rotating the latent factors as L_rot = latent_factors_stacked @ V will yield columns
    # with decreasing variance (S^2 are proportional to the variances).
    R = Vt.T
    
    # Compute the explained variances for each rotated latent dimension.
    explained_variances = (S ** 2) / (latent_factors_stacked.shape[0] - 1)
    
    return R, explained_variances

def explained_variance(observations, emissions):
    """
    Given:
      observations: jnp.ndarray of shape (num_trials, num_timesteps, emission_dim)
      emissions:    jnp.ndarray of shape (num_trials, emission_dim, latent_dim)
    Returns:
      R: jnp.ndarray of shape (latent_dim, latent_dim)
         A rotation matrix such that rotating each trial's emission matrix (i.e. computing O_i @ R)
         orders the latent columns in descending order of explained variance.
      explained_variances: jnp.ndarray of shape (latent_dim,)
         Variances explained by the rotated latent dimensions.
    """
    # Compute latent factors per trial: shape (num_trials, num_timesteps, latent_dim)
    # Using the fact that the emission matrices are orthogonal.
    latent_factors = jnp.einsum('...te,...el->...tl', observations, emissions)
    
    # Stack trials and timesteps together, so we have all latent factors in one 2D array.
    if latent_factors.ndim == 3:
        num_trials, num_timesteps, latent_dim = latent_factors.shape
    else:
        num_timesteps, latent_dim = latent_factors.shape
        num_trials = 1
    latent_factors_stacked = latent_factors.reshape(num_trials * num_timesteps, latent_dim)
    
    # Perform SVD on the aggregated latent factors.
    # latent_factors_stacked = U @ diag(S) @ Vt, with singular values S in descending order.
    U, S, Vt = jnp.linalg.svd(latent_factors_stacked, full_matrices=False)
    
    # Compute the explained variances for each rotated latent dimension.
    explained_variances = (S ** 2) / (latent_factors_stacked.shape[0] - 1)
    
    return explained_variances

def compute_explained_variance(y, full_H, H):
    ev = explained_variance(y, H)
    total_ev = explained_variance(y, full_H).sum()
    return ev / total_ev

def compute_cos_dist(a, b):
    def _l2(x):
        return jnp.sqrt(jnp.sum(jnp.square(x)))
    return 1 - (a @ b) / (_l2(a) * _l2(b))

def compute_cos_sim(a, b):
    def _l2(x):
        return jnp.sqrt(jnp.sum(jnp.square(x)))
    return (a @ b) / (_l2(a) * _l2(b))

def plot_eigenvalues(
    true_A: np.ndarray,
    smds_A: np.ndarray,
    lds_A: np.ndarray,
    figsize: tuple = (4, 4),
    title: str = "Eigenvalues of Dynamics Matrix",
):
    """
    Plot eigenvalues of two dynamics matrices in the complex plane.

    Parameters
    ----------
    A_stationary : np.ndarray
        Dynamics matrix expected to be (asymptotically) stable
        (eigenvalues inside or on the unit circle).
    A_nonstationary : np.ndarray
        Dynamics matrix that may be unstable or marginally stable.
    figsize : tuple, optional
        Size of the matplotlib figure.
    title : str, optional
        Title of the plot.
    """
    # --- compute eigenvalues -------------------------------------------------
    eig_true = np.linalg.eigvals(true_A)
    eig_smds = np.linalg.eigvals(smds_A)
    eig_lds = np.linalg.eigvals(lds_A)

    # --- set up figure -------------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Real Part")
    ax.set_ylabel("Imaginary Part")
    ax.grid(True, linestyle="--", alpha=0.5)

    # Equal aspect so the unit circle looks like a circle
    ax.set_aspect("equal", adjustable="box")

    # --- unit circle ---------------------------------------------------------
    theta = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(theta), np.sin(theta), "k--", lw=1.5)

    # --- scatter eigenvalues -------------------------------------------------
    ax.scatter(
        eig_true.real,
        eig_true.imag,
        c="k",
        marker="o",
        s=70,
        label="Eigenvalues (True SMDS)",
    )
    ax.scatter(
        eig_smds.real,
        eig_smds.imag,
        c="red",
        marker="o",
        label="Eigenvalues (Learned SMDS)",
    )

    ax.scatter(
        eig_lds.real,
        eig_lds.imag,
        c="blue",
        marker="o",
        label="Eigenvalues (Learned LDS)",
    )

    ax.legend(loc="upper left", fontsize=8)
    plt.show()

def moving_average_adaptive(x, window_size):
    kernel = jnp.ones(window_size)

    # Numerator: regular convolution
    smoothed = jnp.convolve(x, kernel, mode='same')

    # Denominator: convolution with ones to count valid entries
    counts = jnp.convolve(jnp.ones_like(x), kernel, mode='same')

    return smoothed / counts

In [ ]:
true_state_dim = 2
emission_dim = 10#24
num_trials = 750#750
num_conditions = 4
num_timesteps = 30

# smoothing_window = 100

dof = true_state_dim * (emission_dim - true_state_dim) + true_state_dim * (true_state_dim - 1) // 2

true_model = StiefelManifoldSSM(state_dim=true_state_dim, 
                                emission_dim=emission_dim,
                                num_trials=num_trials,
                                num_conditions=num_conditions)

key = jr.PRNGKey(123)
# key = jr.PRNGKey(31231)
dynamics = random_rotation(seed=key, n=true_state_dim, theta=jnp.pi/5)

key, key_root = jr.split(key)
true_base_subspace = jr.orthogonal(key_root, emission_dim)

key, key_root = jr.split(key)
true_tau = jr.uniform(key_root, shape=(dof,), minval=1e-8, maxval=1e-4)
# true_tau = true_tau.at[0].set(1e-4)
print(true_tau)

_velocity_cov = jnp.diag(true_tau)
def _get_velocity(prev_velocity, current_key):
    current_velocity_dist = MVN(loc=prev_velocity, covariance_matrix=_velocity_cov)
    current_velocity = current_velocity_dist.sample(seed=current_key)
    return current_velocity, current_velocity

keys = jr.split(key, num_trials)
key = keys[-1]
key, key_root = jr.split(key)
_initial_velocity = jnp.zeros(dof)
_, _velocity = jax.lax.scan(_get_velocity, _initial_velocity, keys[:-1])
_velocity = jnp.concatenate([_initial_velocity[None], _velocity])
true_tau = jnp.var(jnp.diff(_velocity, axis=0), axis=0)

key, key_root = jr.split(key)
true_params, param_props, true_velocity = true_model.initialize(tau=true_tau,
                                                                base_subspace=true_base_subspace,
                                                                key=key, 
                                                                initial_mean=jnp.sqrt(emission_dim/true_state_dim)*jr.normal(key_root, shape=(num_conditions, true_state_dim)),
                                                                dynamics_weights=dynamics,
                                                                # dynamics_bias=jr.normal(key_root, true_state_dim),
                                                                dynamics_covariance=jnp.eye(true_state_dim)*1e-2,
                                                                emission_covariance=jnp.eye(emission_dim)*1e-2,
                                                                velocity=_velocity,
                                                                )

conditions = jnp.tile(jnp.arange(num_conditions), num_trials)[:num_trials]
key, key_root = jr.split(key)
true_states, emissions, _ = true_model.sample(true_params, key, num_timesteps, conditions=conditions)

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# from matplotlib.ticker import MaxNLocator, LinearLocator

# trial_idx = np.array([10, 500])


# # Create a figure and a 3D subplot
# fig = plt.figure(figsize=(16, 8))
# ax = fig.add_subplot(111, projection='3d')

# # Sample 3D data (e.g., a parametric curve like in the image)
# t = np.linspace(-4 * np.pi, 4 * np.pi, 200)
# x_data = np.sin(t) * np.cos(t*0.5)
# y_data = np.sin(t) * np.sin(t*0.5)
# z_data = np.cos(t)

# # # Plot the 3D curve
# # ax.plot(x_data, y_data, z_data, color='sandybrown', linewidth=3, alpha=0.8)
# for idx in trial_idx:
#     ax.plot(*emissions[idx].T, label=f'Trial {idx}', linewidth=2)
# # # Add a darker segment for contrast, similar to the image
# # ax.plot(x_data[:50], y_data[:50], z_data[:50], color='black', linewidth=3.5)


# # --- Customization for axes and grid ---

# # 1. Set the color of the panes (the "walls" of the 3D box)
# pane_color_rgba = (1.0, 1.0, 1.0, 1.0) # White, Opaque
# ax.xaxis.set_pane_color(pane_color_rgba)
# ax.yaxis.set_pane_color(pane_color_rgba)
# ax.zaxis.set_pane_color(pane_color_rgba)

# # 2. Set the color and style of the pane edges (the lines forming the box)
# pane_edge_color = 'white'
# pane_edge_linewidth = 1.5 # Thicker edges for the box
# ax.xaxis.pane.set_edgecolor(pane_edge_color)
# ax.yaxis.pane.set_edgecolor(pane_edge_color)
# ax.zaxis.pane.set_edgecolor(pane_edge_color)
# ax.xaxis.pane.set_linewidth(pane_edge_linewidth)
# ax.yaxis.pane.set_linewidth(pane_edge_linewidth)
# ax.zaxis.pane.set_linewidth(pane_edge_linewidth)

# # 3. Explicitly set tick locators to define where grid lines should be drawn
# # This ensures there are ticks for the grid, even if we make them invisible later.
# # Adjust the number of bins (e.g., 5-7) to control grid density.
# num_grid_lines = 6
# # ax.xaxis.set_major_locator(MaxNLocator(nbins=num_grid_lines))
# # ax.yaxis.set_major_locator(MaxNLocator(nbins=num_grid_lines))
# # ax.zaxis.set_major_locator(MaxNLocator(nbins=num_grid_lines))
# ax.xaxis.set_major_locator(LinearLocator(numticks=num_grid_lines))
# ax.yaxis.set_major_locator(LinearLocator(numticks=num_grid_lines))
# ax.zaxis.set_major_locator(LinearLocator(numticks=num_grid_lines))

# # 4. Configure the grid lines using _axinfo for robust styling
# # This method directly modifies the properties Matplotlib uses for drawing.
# grid_line_color = 'dimgray'
# grid_line_width = 0.7 # Thinner than pane edges, adjust as needed

# for axis_obj in [ax.xaxis, ax.yaxis, ax.zaxis]:
#     axis_obj._axinfo["grid"].update({
#         'color': grid_line_color,
#         'linewidth': grid_line_width,
#         'linestyle': '-'
#     })
#     # Ensure grid is on (though setting color/linewidth usually implies this)
#     # If you had `ax.grid(False)` earlier, ensure this overrides it.
#     # For _axinfo, setting the style IS enabling it.

# # 5. Remove axis tick marks and labels visually
# # (AFTER setting locators and styling grid, as grid relies on tick positions)

# # Remove tick labels
# ax.set_xticklabels([])
# ax.set_yticklabels([])
# ax.set_zticklabels([])

# # ax.set_xlabel('Neuron 1')
# # ax.set_ylabel('Neuron 2')
# # ax.set_zlabel('Neuron 3')

# # Make tick marks invisible (zero length)
# ax.tick_params(axis='x', length=0)
# ax.tick_params(axis='y', length=0)
# ax.tick_params(axis='z', length=0)
# # Alternatively, to remove tick marks completely (might also remove grid if not careful with locators):
# # ax.set_xticks([])
# # ax.set_yticks([])
# # ax.set_zticks([])
# # Using tick_params with length=0 is often safer when you want grids but no visible ticks.

# # 6. (Optional) Set axis limits if needed, otherwise, Matplotlib will autoscale
# # ax.set_xlim([-1.1, 1.1]) # Ensure data is within limits
# # ax.set_ylim([-1.1, 1.1])
# # ax.set_zlim([-1.1, 1.1])

# # 7. (Optional) Set the view angle
# ax.view_init(elev=20, azim=20) # Adjust elevation and azimuth
# plt.savefig("true_observation.png", bbox_inches='tight', facecolor='white', dpi=600)
# plt.show()

In [ ]:
true_tau.max()

In [ ]:
D = state_dim=true_params.initial.mean.shape[-1]
N = emission_dim=true_params.emissions.weights.shape[-2]
num_trials=len(true_params.emissions.weights)
num_conditions = true_params.initial.mean.shape[0]

num_timesteps=emissions.shape[-2]

dof = state_dim * (emission_dim - state_dim)
dof_shape = (state_dim, emission_dim - state_dim)

In [ ]:
# Plot the true states and emissions
fig, ax = plt.subplots(figsize=(3, 4))
ax.plot(emissions[-1, :, :min(10, N)] + 1 * jnp.arange(min(10, N)))
ax.set_ylabel("data")
ax.set_xlabel("time")
ax.set_xlim(0, num_timesteps - 1)
plt.show()

In [ ]:
true_emissions_weights = true_params.emissions.weights
print(true_emissions_weights.shape)

In [ ]:
true_corrcoef = jnp.corrcoef(true_emissions_weights.reshape(len(emissions), -1))
plt.figure(figsize=(3,2.5))
plt.imshow(true_corrcoef, aspect='auto', interpolation='none')
# plt.colorbar(label='Emission matrix correlation')
plt.ylabel('Trials')
plt.xlabel('Trials')
# plt.yticks(np.array([0, 750, 1500])-0.5, [1, 750, 1500])
# plt.xticks(np.array([0, 750, 1500])-0.5, [1, 750, 1500])
plt.title('SMDS\nGround truth drift')
plt.colorbar(label='correlation')
# plt.savefig('simulated_true_C_corrcoef_drift.pdf', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
subspace_angle_matrix = compute_subspace_angle_matrix_v2(true_emissions_weights[:, :, :])
normalized_grassmann_dist_matrix = jnp.linalg.norm(subspace_angle_matrix, ord=2, axis=-1) / (jnp.sqrt(state_dim) * jnp.pi / 2)

In [ ]:
plt.figure(figsize=(3,2.5))
# plt.imshow(normalized_grassmann_dist_matrix, interpolation='None', vmin=0.0, vmax=0.3615, cmap='inferno')
plt.imshow(normalized_grassmann_dist_matrix, interpolation='None', vmin=0.0, cmap='inferno')
# plt.colorbar(label='Grassmann Distance\nNormalized to [0,1]')
plt.colorbar(label=r'Grassmann Distance $\in$ [0,1]')
plt.title(f'SMDS\nGround Truth Drift')
plt.xticks([0, 250, 500, 749], [1, 200, 500, 750])
plt.yticks([0, 250, 500, 749], [1, 200, 500, 750])
# plt.xticks([0, 20//block_size, 40//block_size, 60//block_size, 80//block_size, 100//block_size, 120//block_size],
#            [1, 20, 40, 60, 80, 100, 120])
# plt.yticks([0, 20//block_size, 40//block_size, 60//block_size, 80//block_size, 100//block_size, 120//block_size],
#            [1, 20, 40, 60, 80, 100, 120])
plt.ylabel('Trials')
plt.xlabel('Trials')
# plt.savefig('./figure2/figure2_smds_true_drift_final_v2.png', bbox_inches='tight', facecolor='white', dpi=600)
plt.show()

In [ ]:
# twist_angle_matrix = compute_twist_angle_matrix(true_emissions_weights[:, :, :])
twist_angle_matrix = compute_twist_angle_matrix(torch.tensor(np.array(true_emissions_weights, copy=True), device='cuda'))
twist_angle_matrix = jnp.array(twist_angle_matrix.cpu().numpy())
d_in_max = jnp.pi * jnp.sqrt(D // 2)
normalized_twist_angle_matrix = jnp.clip(twist_angle_matrix / d_in_max, 0.0, 1.0)

In [ ]:
plt.figure(figsize=(3,2.5))
# plt.imshow(normalized_grassmann_dist_matrix, interpolation='None', vmin=0.0, vmax=0.3615, cmap='inferno')
plt.imshow(normalized_twist_angle_matrix, interpolation='None', vmin=0.0, cmap='inferno')
# plt.colorbar(label='Grassmann Distance\nNormalized to [0,1]')
plt.colorbar(label=r'Within Subspace Rotation $\in$ [0,1]')
plt.title(f'SMDS\nGround Truth Drift')
plt.xticks([0, 250, 500, 749], [1, 200, 500, 750])
plt.yticks([0, 250, 500, 749], [1, 200, 500, 750])
# plt.xticks([0, 20//block_size, 40//block_size, 60//block_size, 80//block_size, 100//block_size, 120//block_size],
#            [1, 20, 40, 60, 80, 100, 120])
# plt.yticks([0, 20//block_size, 40//block_size, 60//block_size, 80//block_size, 100//block_size, 120//block_size],
#            [1, 20, 40, 60, 80, 100, 120])
plt.ylabel('Trials')
plt.xlabel('Trials')
# plt.savefig('./figure2/figure2_smds_true_drift_final_v2.png', bbox_inches='tight', facecolor='white', dpi=600)
plt.show()

In [ ]:
true_rotation, _ = compute_rotation(emissions, true_params.emissions.weights)
A = true_params.dynamics.weights
true_rotation_inv = jnp.linalg.inv(true_rotation)
transformed_true_A = true_rotation @ A @ true_rotation_inv

plot_eigenvalues(transformed_true_A, transformed_true_A, transformed_true_A, figsize=(8,8))

In [ ]:
# angles = jnp.zeros((num_trials, num_trials, D))
# for r in tqdm(range(num_trials)):
#     a = vmap(subspace_angles, in_axes=(None, 0))(true_emissions_weights[r], true_emissions_weights[r:])
#     angles = angles.at[r, r:].set(a)

In [ ]:
# grassmann_distance = jnp.sqrt(jnp.sum(jnp.square(angles), axis=-1))
# normalized_grassmann_distance = grassmann_distance / (jnp.pi * jnp.sqrt(state_dim) * 0.5)

In [ ]:
true_emissions_weights = true_params.emissions.weights

In [ ]:
R, explained_variances = compute_rotation(emissions, true_emissions_weights)
rotated_true_emissions_weights = jnp.einsum('rij,jk->rik', true_emissions_weights, R)
true_evs = vmap(compute_explained_variance, in_axes=(None, None, -1))(emissions, rotated_true_emissions_weights, rotated_true_emissions_weights[:, :, None])
original_true_evs = vmap(compute_explained_variance, in_axes=(None, None, -1))(emissions, true_emissions_weights, 
                                                                               true_emissions_weights[:, :, None])

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(D), true_evs)
ax.set_ylabel('explained variance')
ax.set_xlabel('state dim')
plt.show()

In [ ]:
true_cos_sims = np.zeros((D, num_trials))
true_cos_dists = np.zeros((D, num_trials))
for d in tqdm(range(D)):
    true_cos_sims[d] = vmap(compute_cos_sim, in_axes=(0, None))(true_emissions_weights[:,:,d], 
                                                                  true_emissions_weights[0,:,d])
    true_cos_dists[d] = vmap(compute_cos_dist, in_axes=(0, None))(true_emissions_weights[:,:,d], 
                                                                    true_emissions_weights[0,:,d])

In [ ]:
plt.figure(figsize=(10,8))
for d in range(D):
    plt.plot(true_cos_dists[d], label=f'explains {100*original_true_evs[d, 0]:.2f}% of variance')
plt.ylabel('cosine distance')
plt.xlabel('trial')
plt.legend()
plt.show()

In [ ]:
rotated_true_cos_sims = np.zeros((D, num_trials))
rotated_true_cos_dists = np.zeros((D, num_trials))
for d in tqdm(range(D)):
    rotated_true_cos_sims[d] = vmap(compute_cos_sim, in_axes=(0, None))(rotated_true_emissions_weights[:,:,d], 
                                                                          rotated_true_emissions_weights[0,:,d])
    rotated_true_cos_dists[d] = vmap(compute_cos_dist, in_axes=(0, None))(rotated_true_emissions_weights[:,:,d], 
                                                                          rotated_true_emissions_weights[0,:,d])

In [ ]:
plt.figure(figsize=(10,8))
for d in range(D):
    plt.plot(rotated_true_cos_dists[d], label=f'explains {100*true_evs[d, 0]:.2f}% of variance')
plt.ylabel('cosine distance')
plt.xlabel('trial')
plt.legend()
plt.show()

In [ ]:
rotated_true_cos_sims_diag = np.zeros((D, num_trials-1))
rotated_true_cos_dists_diag = np.zeros((D, num_trials-1))
for d in tqdm(range(D)):
    rotated_true_cos_sims_diag[d] = vmap(compute_cos_sim, in_axes=(0, 0))(rotated_true_emissions_weights[:-1,:,d], 
                                                                          rotated_true_emissions_weights[1:,:,d])
    rotated_true_cos_dists_diag[d] = vmap(compute_cos_dist, in_axes=(0, 0))(rotated_true_emissions_weights[:-1,:,d], 
                                                                            rotated_true_emissions_weights[1:,:,d])

In [ ]:
true_diag_thetas = np.rad2deg(jnp.arccos(jnp.clip(rotated_true_cos_sims_diag, min=-1, max=1)))
true_diag_thetas = np.array(true_diag_thetas)

In [ ]:
plt.figure(figsize=(4,3))
plt.scatter(true_evs, true_diag_thetas.mean(1), s=10)
plt.title('Drift Rate vs. Explained Variance Per Axis')

plt.xlabel('Explained variance')
plt.ylabel('mean(' + r'$\Delta\theta$' + ' per trial) in degrees')
plt.show()

In [ ]:
block_size = 1
num_blocks = len(emissions) // block_size
print(num_blocks)
num_trials = num_blocks * block_size
emissions = emissions[:num_trials]
conditions = conditions[:num_trials]

block_masks = jnp.ones(num_blocks, dtype=bool)
trial_masks = jnp.ones(len(emissions), dtype=bool)
num_test_blocks = num_blocks // 6
key = jr.PRNGKey(2626)
# key = jr.PRNGKey(66123)

test_idx = jax.random.choice(key, jnp.arange(30, num_blocks-30, dtype=int), shape=(num_test_blocks,), replace=False)

block_masks = block_masks.at[test_idx].set(False)
num_train_blocks = block_masks.sum()
print(num_train_blocks)
block_ids = jnp.repeat(jnp.eye(num_blocks), block_size, axis=1)
block_id_nums = jnp.repeat(jnp.arange(num_blocks, dtype=float), block_size) / (num_blocks - 1)
print(block_id_nums.min(), block_id_nums.max())
assert np.isclose(float(block_id_nums.min()), 0.0)
assert np.isclose(float(block_id_nums.max()), 1.0)

trial_masks = jnp.repeat(block_masks, block_size)

train_obs = emissions
_, sequence_length, emission_dim = train_obs.shape
test_obs = train_obs[~trial_masks]

train_conditions = conditions[trial_masks]
test_conditions = conditions[~trial_masks]

print(len(jnp.unique(train_conditions)))

## CLDS setup


In [ ]:
# CLDS helpers
CLDS_BASIS_TYPE = 'fourier'
CLDS_L = 5
CLDS_SIGMA = 0.5
CLDS_KAPPA = 0.6
CLDS_PERIOD = 1.0 + 6.0 * CLDS_KAPPA
CLDS_COLOR = np.array([0, 130, 80]) / 255


def make_clds_basis(L=CLDS_L, sigma=CLDS_SIGMA, kappa=CLDS_KAPPA):
    period = 1.0 + 6.0 * kappa
    return Tm_basis(L, M_conditions=1, sigma=sigma, kappa=kappa, period=period)


def make_clds_model(state_dim, basis_funcs=None, has_dynamics_bias=False):
    if basis_funcs is None:
        basis_funcs = make_clds_basis()
    return ConditionallyLinearGaussianSSM(
        state_dim=state_dim,
        emission_dim=emission_dim,
        num_conditions=num_conditions,
        has_dynamics_bias=has_dynamics_bias,
        has_emissions_bias=False,
        torus_basis_funcs=basis_funcs,
        num_trials=len(train_obs[trial_masks]),
    )


def initialize_clds_with_constant_pca(clds_model, key, state_dim):
    base_subspace = PCA(n_components=emission_dim).fit(
        train_obs[trial_masks].reshape(-1, emission_dim)
    ).components_.T
    emission_weights = jnp.zeros((clds_model.wpgs_C.L, emission_dim, state_dim))
    phi0 = clds_model.wpgs_C.evaluate_basis(0.0)[0]
    emission_weights = emission_weights.at[0].set(base_subspace[:, :state_dim] / phi0)
    return clds_model.initialize(key=key, emission_weights=emission_weights)


def fit_clds_for_dim(test_state_dim, key, num_iters=200):
    clds_model = make_clds_model(test_state_dim)
    clds_params, clds_props = initialize_clds_with_constant_pca(clds_model, key, test_state_dim)
    clds_params, clds_marginal_lls = clds_model.fit_em(
        params=clds_params,
        props=clds_props,
        emissions=train_obs[trial_masks],
        conditions=train_conditions,
        block_id_nums=block_id_nums[trial_masks],
        num_iters=num_iters,
        use_wandb=False,
    )
    clds_test_marginal_ll = clds_model.batch_marginal_log_prob(
        clds_params,
        test_obs,
        conditions=test_conditions,
        trial_ids=block_id_nums[~trial_masks],
    )
    return clds_model, clds_params, clds_marginal_lls, clds_test_marginal_ll


print(f'CLDS basis: {CLDS_BASIS_TYPE}, L={CLDS_L}, sigma={CLDS_SIGMA}, kappa={CLDS_KAPPA}, period={CLDS_PERIOD:.2f}')
print(f'block_id_nums range: {float(block_id_nums.min()):.3f} to {float(block_id_nums.max()):.3f}')


In [ ]:
plt.figure(figsize=(24,2))
for i in np.where(~trial_masks)[0]:
    plt.axvspan(i, i+1, alpha=0.5, color='r')
plt.xlim(0, len(emissions))
# plt.ylabel('Geodesic distance')
plt.xlabel('Trials')
plt.show()

In [ ]:
xy_ekf_marginal_ll = true_model.marginal_log_prob(true_params, train_obs.reshape(num_blocks, block_size, sequence_length, emission_dim), 
                                                  conditions=conditions.reshape(num_blocks, block_size), block_masks=jnp.ones(num_blocks, dtype=bool),
                                                  trial_masks=jnp.ones((num_blocks, block_size), dtype=bool),
                                                  method=1, num_iters=1)
y_ekf_marginal_ll = true_model.marginal_log_prob(true_params, train_obs.reshape(num_blocks, block_size, sequence_length, emission_dim), 
                                                 conditions=conditions.reshape(num_blocks, block_size), block_masks=block_masks, 
                                                 trial_masks=trial_masks.reshape(num_blocks, block_size),
                                                 method=1, num_iters=1)
true_ekf_marginal_ll = xy_ekf_marginal_ll - y_ekf_marginal_ll
print(true_ekf_marginal_ll)

In [ ]:
stationary_test_state_dims = np.arange(1, 11, 1)
num_iters = 200
key = jr.PRNGKey(123456)
test_marginal_lls = []
for test_state_dim in stationary_test_state_dims:
    test_model = LinearGaussianConjugateSSM(test_state_dim, emission_dim, 
                                            num_conditions=num_conditions,
                                            has_dynamics_bias=False, has_emissions_bias=False)
            
    key, this_key = jr.split(key, 2)

    base_subspace = PCA(n_components=emission_dim).fit(train_obs[trial_masks].reshape(-1, emission_dim)).components_.T
    emission_weights = base_subspace[:, :test_state_dim]
    
    test_params, param_props = test_model.initialize(key=this_key, 
                                                     emission_weights=emission_weights
                                                    )
    
    test_params, marginal_lls = test_model.fit_em(test_params, 
                                                  param_props, 
                                                  train_obs[trial_masks],
                                                  conditions=train_conditions,
                                                  num_iters=num_iters)
    
    plt.figure(figsize=(1,1))
    plt.plot(marginal_lls)
    plt.show()

    plt.figure(figsize=(1,1))
    plt.plot(marginal_lls[1:])
    plt.show()

    plt.figure(figsize=(1,1))
    plt.plot(marginal_lls[100:])
    plt.show()

    print(test_state_dim, marginal_lls[-1])
    print(monotonically_increasing(jnp.array(marginal_lls), rtol=1e-2, atol=1e-2))
    print(jnp.diff(jnp.array(marginal_lls)).min())

    test_marginal_ll = test_model.batch_marginal_log_prob(test_params, test_obs, conditions=test_conditions)

    print(jnp.diag(test_params.emissions.cov).min())
    
    test_marginal_lls.append(test_marginal_ll)
    print(test_state_dim, test_marginal_ll)

In [ ]:
plt.axhline(true_ekf_marginal_ll)
plt.plot(stationary_test_state_dims, test_marginal_lls)

In [ ]:
# run this
key = jr.PRNGKey(123456)
test_state_dims = [1, 2, 3, 4]
test_ekf_marginal_lls = []
num_iters = 200
for test_state_dim in test_state_dims:
    ddof = test_state_dim * (emission_dim - test_state_dim) + test_state_dim * (test_state_dim - 1) // 2
    # dof = (test_state_dim, emission_dim - test_state_dim)
    test_model = StiefelManifoldSSM(test_state_dim, 
                                    emission_dim,
                                    num_trials=len(train_obs),
                                    num_conditions=num_conditions,
                                    has_dynamics_bias=False,
                                    tau_per_dim=True,
                                    tau_per_axis=False,
                                    fix_tau=False,
                                    fix_initial_velocity=False,
                                    fix_scale=True,
                                    emissions_cov_eps=1e-5,
                                    velocity_smoother_method='ekf',
                                    ekf_mode='cov',
                                    max_tau=1e-3,
                                    ekf_num_iters=1,
                                    initial_velocity_covariance_prior=IG(concentration=1e-6, scale=1e-6),
                                    tau_prior=IG(concentration=1e-6, scale=1e-6),
                                    )

    base_subspace = PCA(n_components=emission_dim).fit(train_obs[trial_masks].reshape(-1, emission_dim)).components_.T
    emission_weights = jnp.tile(base_subspace[:, :test_state_dim][None], (len(train_obs), 1, 1))
    
    key, key_root = jr.split(key)
    test_params, param_props, _ = test_model.initialize(base_subspace=base_subspace,
                                                tau=jnp.ones(ddof)*1e-6,
                                                emission_weights=emission_weights,
                                                initial_velocity_cov=1e-4*jnp.eye(ddof),
                                                key=key)
    
    test_params, marginal_lls = test_model.fit_em(test_params, param_props, 
                                                  train_obs,
                                                  conditions=conditions,
                                                  trial_masks=trial_masks,
                                                  block_ids=block_ids,
                                                  block_masks=block_masks,
                                                  num_iters=num_iters,
                                                  print_ll=True,
                                                  run_velocity_smoother=False)

    plt.figure(figsize=(1,1))
    plt.plot(marginal_lls)
    plt.show()
    
    print(jnp.diff(jnp.array(marginal_lls)).min(), jnp.argmax(marginal_lls))

    start_time = time.time()
    xy_ekf_marginal_ll = test_model.marginal_log_prob(test_params, train_obs.reshape(num_blocks, block_size, sequence_length, emission_dim), 
                                                      conditions=conditions.reshape(num_blocks, block_size), block_masks=jnp.ones(num_blocks, dtype=bool),
                                                      trial_masks=jnp.ones((num_blocks, block_size), dtype=bool),
                                                      method=1, num_iters=1)
    y_ekf_marginal_ll = test_model.marginal_log_prob(test_params, train_obs.reshape(num_blocks, block_size, sequence_length, emission_dim), 
                                                     conditions=conditions.reshape(num_blocks, block_size), block_masks=block_masks, 
                                                     trial_masks=trial_masks.reshape(num_blocks, block_size),
                                                     method=1, num_iters=1)
    test_ekf_marginal_ll = xy_ekf_marginal_ll - y_ekf_marginal_ll

    end_time = time.time()  # Record the end time
    
    elapsed_time = end_time - start_time  # Calculate the elapsed time in seconds
    print(f"Elapsed time: {elapsed_time:.6f} seconds")  # Print with microsecond precision    
    test_ekf_marginal_lls.append(test_ekf_marginal_ll)

    print(test_state_dim, test_ekf_marginal_ll)

In [ ]:
test_marginal_lls = np.array(test_marginal_lls)
test_ekf_marginal_lls = np.array(test_ekf_marginal_lls)

In [ ]:
num_test_samples = (num_trials // 6) * N * num_timesteps

In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

fig, ax = plt.subplots(figsize=(3,2.5))
plt.grid(True, zorder=0)

ax.axhline(true_ekf_marginal_ll / num_test_samples, color='k', linestyle=':', label='True SMDS', zorder=0, linewidth=2)

ax.plot(test_state_dims, test_ekf_marginal_lls / num_test_samples, marker='o', color='r', label='SMDS', zorder=3, linewidth=2)

ax.plot(stationary_test_state_dims, test_marginal_lls / num_test_samples, marker='o', color='b', label='LDS', zorder=3, linewidth=2)


ax.axvline(2, color='k', linestyle='-', zorder=2, linewidth=2)
ax.text(2.2, -0.1, 'True Dim.', rotation=90, fontsize=10)
# ax.legend(loc='lower left')
ax.set_xlabel('Latent State Dim.')
ax.set_ylabel('Normalized\nTest Log-likelihood')
ax.set_xticks(stationary_test_state_dims)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.05), fontsize=10)

plt.savefig('figure2_clds/figure2_test_ll.pdf', 
            bbox_inches='tight', facecolor='white', transparent=True)

plt.show()

## CLDS test log likelihood


In [ ]:
# Fit CLDS models and score held-out test log likelihoods
clds_test_state_dims = list(test_state_dims)
clds_models = {}
clds_train_lps = {}
clds_test_marginal_lls = []

num_iters = 200
key = jr.PRNGKey(123456)
for test_state_dim in clds_test_state_dims:
    key, this_key = jr.split(key)
    clds_model, clds_params, clds_marginal_lls, clds_test_marginal_ll = fit_clds_for_dim(
        test_state_dim,
        this_key,
        num_iters=num_iters,
    )
    clds_models[int(test_state_dim)] = (clds_model, clds_params)
    clds_train_lps[int(test_state_dim)] = clds_marginal_lls
    clds_test_marginal_lls.append(clds_test_marginal_ll)

    plt.figure(figsize=(1, 1))
    plt.plot(clds_marginal_lls)
    plt.title(f'CLDS D={test_state_dim}')
    plt.show()

    print(test_state_dim, clds_marginal_lls[-1], clds_test_marginal_ll)

clds_test_marginal_lls = np.array(clds_test_marginal_lls)


In [ ]:
# Normalized test log-likelihood with CLDS
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

fig, ax = plt.subplots(figsize=(3.2, 2.5))
plt.grid(True, zorder=0)

ax.axhline(true_ekf_marginal_ll / num_test_samples, color='k', linestyle=':', label='True SMDS', zorder=0, linewidth=2)
ax.plot(test_state_dims, test_ekf_marginal_lls / num_test_samples, marker='o', color='r', label='SMDS', zorder=3, linewidth=2)
ax.plot(stationary_test_state_dims, test_marginal_lls / num_test_samples, marker='o', color='b', label='LDS', zorder=3, linewidth=2)
ax.plot(clds_test_state_dims, clds_test_marginal_lls / num_test_samples, marker='o', color=CLDS_COLOR, label='CLDS', zorder=3, linewidth=2)

ax.axvline(D, color='k', linestyle='-', zorder=2, linewidth=2)
ax.text(D + 0.2, -0.1, 'True Dim.', rotation=90, fontsize=10)
ax.set_xlabel('Latent State Dim.')
ax.set_ylabel('Normalized\nTest Log-likelihood')
ax.set_xticks(stationary_test_state_dims)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.05), fontsize=9)

plt.savefig('figure2_clds/figure2_test_ll_with_clds.pdf', bbox_inches='tight', facecolor='white', transparent=True)
plt.show()


In [ ]:
num_iters = 200
key = jr.PRNGKey(123456)
lds_models = []
for test_state_dim in [2, 3]:
    test_model = LinearGaussianConjugateSSM(test_state_dim, emission_dim, 
                                            num_conditions=num_conditions,
                                            has_dynamics_bias=False, has_emissions_bias=False)
            
    key, this_key = jr.split(key, 2)

    base_subspace = PCA(n_components=emission_dim).fit(train_obs.reshape(-1, emission_dim)).components_.T
    emission_weights = base_subspace[:, :test_state_dim]
    
    test_params, param_props = test_model.initialize(key=this_key, 
                                                     emission_weights=emission_weights
                                                    )
    
    test_params, marginal_lls = test_model.fit_em(test_params, 
                                                  param_props, 
                                                  train_obs,
                                                  conditions=conditions,
                                                  num_iters=num_iters)
    
    plt.figure(figsize=(1,1))
    plt.plot(marginal_lls)
    plt.show()

    plt.figure(figsize=(1,1))
    plt.plot(marginal_lls[1:])
    plt.show()

    plt.figure(figsize=(1,1))
    plt.plot(marginal_lls[100:])
    plt.show()

    print(test_state_dim, marginal_lls[-1])
    print(monotonically_increasing(jnp.array(marginal_lls), rtol=1e-2, atol=1e-2))
    print(jnp.diff(jnp.array(marginal_lls)).min())

    lds_models.append((test_model, test_params))

In [ ]:
# run this
key = jr.PRNGKey(123456)
smds_models = []
num_iters = 200
for test_state_dim in [2]:
    ddof = test_state_dim * (emission_dim - test_state_dim) + test_state_dim * (test_state_dim - 1) // 2
    test_model = StiefelManifoldSSM(test_state_dim, 
                                    emission_dim,
                                    num_trials=len(train_obs),
                                    num_conditions=num_conditions,
                                    has_dynamics_bias=False,
                                    tau_per_dim=True,
                                    tau_per_axis=False,
                                    fix_tau=False,
                                    fix_initial_velocity=False,
                                    fix_scale=True,
                                    emissions_cov_eps=1e-5,
                                    velocity_smoother_method='ekf',
                                    ekf_mode='cov',
                                    max_tau=1e-3,
                                    ekf_num_iters=1,
                                    initial_velocity_covariance_prior=IG(concentration=1e-6, scale=1e-6),
                                    tau_prior=IG(concentration=1e-6, scale=1e-6),
                                    )

    base_subspace = PCA(n_components=emission_dim).fit(train_obs.reshape(-1, emission_dim)).components_.T
    emission_weights = jnp.tile(base_subspace[:, :test_state_dim][None], (len(train_obs), 1, 1))
    
    key, key_root = jr.split(key)
    test_params, param_props, _ = test_model.initialize(base_subspace=base_subspace,
                                                tau=jnp.ones(ddof)*1e-6,
                                                emission_weights=emission_weights,
                                                initial_velocity_cov=1e-4*jnp.eye(ddof),
                                                key=key)
    
    test_params, marginal_lls = test_model.fit_em(test_params, param_props, 
                                                  train_obs,
                                                  conditions=conditions,
                                                  num_iters=num_iters,
                                                  print_ll=True,
                                                  run_velocity_smoother=False)

    plt.figure(figsize=(1,1))
    plt.plot(marginal_lls)
    plt.show()
    
    print(jnp.diff(jnp.array(marginal_lls)).min(), jnp.argmax(marginal_lls))
    smds_models.append((test_model, test_params))

In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# trial_idx = np.array([10, 350, 740])
# trial_idx = np.array([10, 500, 745])
trial_idx = np.array([1, 402])

# vf_x = lambda x, y: y
# vf_y = lambda x, y: -x
A = true_params.dynamics.weights
def vf_x(x, y):
    point = jnp.array([x, y])
    return A[0] @ point - x

def vf_y(x, y):
    point = jnp.array([x, y])
    return A[1] @ point - y

x_lim = (-4, 4)
y_lim = (-4, 4)

step = 0.75#0.25
scale = 6

X, Y = np.meshgrid(np.arange(x_lim[0], x_lim[1], step), np.arange(y_lim[0], y_lim[1], step))
U = np.zeros(X.shape)
V = np.zeros(Y.shape)

for i in range(X.shape[0]):
    for j in range(Y.shape[0]):
          U[i,j] = vf_x(X[i, j], Y[i, j])
          V[i,j] = vf_y(X[i, j], Y[i, j])
          
fig, ax = plt.subplots(figsize=(3, 3))
_ = ax.quiver(X, Y, U, V, units='xy', scale=scale, color='k')

plt.xlim(x_lim)
plt.ylim(y_lim)

# ax.set_xticks(np.arange(x_lim[0], x_lim[1], 1))
# ax.set_yticks(np.arange(y_lim[0], y_lim[1], 1))
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect('equal')

# Move axis to the middle
ax.spines['left'].set_position('zero')

ax.spines['right'].set_color('none')
ax.yaxis.tick_left()

ax.spines['bottom'].set_position('zero')

ax.spines['top'].set_color('none')
ax.xaxis.tick_bottom()
plt.grid()

# ax.plot(*true_states[trial_idx].T)
for idx in trial_idx:
    ax.plot(*true_states[idx].T, label=f'Trial {idx}')
# plt.title('True dynamics')
plt.legend()
# plt.savefig('true_dynamics_v2.png', bbox_inches='tight', facecolor='white', dpi=600)
plt.savefig('figure2_clds/true_dynamics.pdf', bbox_inches='tight', facecolor='white', transparent=True)
plt.show()

In [ ]:
velocity_smoother1 = smds_models[0][0].smoother(smds_models[0][1], train_obs.reshape(num_blocks, block_size, sequence_length, emission_dim), 
                                                conditions.reshape(num_blocks, block_size), 
                                                jnp.ones(num_blocks, dtype=bool),
                                                method=1, num_iters=1)
Ev1 = velocity_smoother1.smoothed_means

Hs1 = vmap(rotate_subspace, in_axes=(None, None, 0))(smds_models[0][1].emissions.base_subspace, D, Ev1)

mu_0 = smds_models[0][1].initial.mean
Sigma_0 = smds_models[0][1].initial.cov
A = smds_models[0][1].dynamics.weights
b = smds_models[0][1].dynamics.bias
Q = smds_models[0][1].dynamics.cov
R = smds_models[0][1].emissions.cov
emission_scale = smds_models[0][1].emissions.scale
# print(emission_scale)

def batch_smoothed_xs(H, ys, condition):
    trial_test_params = make_lgssm_params(mu_0,
                                          Sigma_0,
                                          A,
                                          Q,
                                          H,
                                          R,
                                          dynamics_bias=b)

    posterior = lgssm_smoother(trial_test_params, ys, None, condition)
    return posterior.smoothed_means

smds_smoothed_xs = vmap(batch_smoothed_xs)(Hs1,
                                           train_obs,
                                           conditions)

In [ ]:
# vf_x = lambda x, y: y
# vf_y = lambda x, y: -x
A = smds_models[0][1].dynamics.weights
def vf_x(x, y):
    point = jnp.array([x, y])
    return A[0] @ point - x

def vf_y(x, y):
    point = jnp.array([x, y])
    return A[1] @ point - y

# x_lim = (-6, 6)
# y_lim = (-6, 6)

step = 0.75#0.25
scale = 6

X, Y = np.meshgrid(np.arange(x_lim[0], x_lim[1], step), np.arange(y_lim[0], y_lim[1], step))
U = np.zeros(X.shape)
V = np.zeros(Y.shape)

for i in range(X.shape[0]):
    for j in range(Y.shape[0]):
          U[i,j] = vf_x(X[i, j], Y[i, j])
          V[i,j] = vf_y(X[i, j], Y[i, j])
          
fig, ax = plt.subplots(figsize=(3, 3))
_ = ax.quiver(X, Y, U, V, units='xy', scale=scale, color='k')

plt.xlim(x_lim)
plt.ylim(y_lim)

# ax.set_xticks(np.arange(x_lim[0], x_lim[1], 1))
# ax.set_yticks(np.arange(y_lim[0], y_lim[1], 1))
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect('equal')

# Move axis to the middle
ax.spines['left'].set_position('zero')

ax.spines['right'].set_color('none')
ax.yaxis.tick_left()

ax.spines['bottom'].set_position('zero')

ax.spines['top'].set_color('none')
ax.xaxis.tick_bottom()
plt.grid()

# ax.plot(*true_states[trial_idx].T)
for idx in trial_idx:
    ax.plot(*smds_smoothed_xs[idx].T, label=f'Trial {idx}')
# plt.title('SMDS Learned Dynamics')
# plt.legend()
# plt.savefig('SMDS_dynamics_v2.png', bbox_inches='tight', facecolor='white', dpi=600)
# plt.savefig('true_dynamics.pdf', bbox_inches='tight', facecolor='white')
plt.savefig('figure2_clds/SMDS_dynamics.pdf', bbox_inches='tight', facecolor='white', transparent=True)
plt.show()

In [ ]:
lds_smoothed_xs_2D = lds_models[0][0].batch_smoother(lds_models[0][1], train_obs, 
                                           conditions=conditions).smoothed_means
lds_smoothed_xs_3D = lds_models[1][0].batch_smoother(lds_models[1][1], train_obs, 
                                           conditions=conditions).smoothed_means

In [ ]:
# vf_x = lambda x, y: y
# vf_y = lambda x, y: -x
A = lds_models[0][1].dynamics.weights
def vf_x(x, y):
    point = jnp.array([x, y])
    return A[0] @ point - x

def vf_y(x, y):
    point = jnp.array([x, y])
    return A[1] @ point - y

# x_lim = (-6, 6)
# y_lim = (-6, 6)

step = 0.75#0.25
scale = 6

X, Y = np.meshgrid(np.arange(x_lim[0], x_lim[1], step), np.arange(y_lim[0], y_lim[1], step))
U = np.zeros(X.shape)
V = np.zeros(Y.shape)

for i in range(X.shape[0]):
    for j in range(Y.shape[0]):
          U[i,j] = vf_x(X[i, j], Y[i, j])
          V[i,j] = vf_y(X[i, j], Y[i, j])
          
fig, ax = plt.subplots(figsize=(3, 3))
_ = ax.quiver(X, Y, U, V, units='xy', scale=scale, color='k')

plt.xlim(x_lim)
plt.ylim(y_lim)

# ax.set_xticks(np.arange(x_lim[0], x_lim[1], 1))
# ax.set_yticks(np.arange(y_lim[0], y_lim[1], 1))
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect('equal')

# Move axis to the middle
ax.spines['left'].set_position('zero')

ax.spines['right'].set_color('none')
ax.yaxis.tick_left()

ax.spines['bottom'].set_position('zero')

ax.spines['top'].set_color('none')
ax.xaxis.tick_bottom()
plt.grid()

# ax.plot(*true_states[trial_idx].T)
for idx in trial_idx:
    ax.plot(*lds_smoothed_xs_2D[idx].T, label=f'Trial {idx}')
# plt.title('SMDS Learned Dynamics')
# plt.legend()
# plt.savefig('SMDS_dynamics_v2.png', bbox_inches='tight', facecolor='white', dpi=600)
# plt.savefig('true_dynamics.pdf', bbox_inches='tight', facecolor='white')
# plt.savefig('SMDS_dynamics_v2.pdf', bbox_inches='tight', facecolor='white', transparent=True)
plt.show()

## CLDS learned dynamics


In [ ]:
# Learned CLDS dynamics and smoothed trajectories
clds_model_2d, clds_params_2d = clds_models[D]
clds_smoothed_xs_2D = clds_model_2d.batch_smoother(
    clds_params_2d,
    train_obs,
    conditions=conditions,
    trial_ids=block_id_nums,
).smoothed_means

A = clds_params_2d.dynamics.weights
def vf_x(x, y):
    point = jnp.array([x, y])
    return A[0] @ point - x


def vf_y(x, y):
    point = jnp.array([x, y])
    return A[1] @ point - y

step = 0.75
scale = 6

X, Y = np.meshgrid(np.arange(x_lim[0], x_lim[1], step), np.arange(y_lim[0], y_lim[1], step))
U = np.zeros(X.shape)
V = np.zeros(Y.shape)

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        U[i, j] = vf_x(X[i, j], Y[i, j])
        V[i, j] = vf_y(X[i, j], Y[i, j])

fig, ax = plt.subplots(figsize=(3, 3))
_ = ax.quiver(X, Y, U, V, units='xy', scale=scale, color='k')

plt.xlim(x_lim)
plt.ylim(y_lim)
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect('equal')
ax.spines['left'].set_position('zero')
ax.spines['right'].set_color('none')
ax.yaxis.tick_left()
ax.spines['bottom'].set_position('zero')
ax.spines['top'].set_color('none')
ax.xaxis.tick_bottom()
plt.grid()

for idx in trial_idx:
    ax.plot(*clds_smoothed_xs_2D[idx].T, label=f'Trial {idx}')

plt.savefig('figure2_clds/CLDS_dynamics.pdf', bbox_inches='tight', facecolor='white', transparent=True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator, LinearLocator

# Create a figure and a 3D subplot
fig = plt.figure(figsize=(16, 8))
ax = fig.add_subplot(111, projection='3d')

# Sample 3D data (e.g., a parametric curve like in the image)
t = np.linspace(-4 * np.pi, 4 * np.pi, 200)
x_data = np.sin(t) * np.cos(t*0.5)
y_data = np.sin(t) * np.sin(t*0.5)
z_data = np.cos(t)

# # Plot the 3D curve
# ax.plot(x_data, y_data, z_data, color='sandybrown', linewidth=3, alpha=0.8)
for idx in trial_idx:
    ax.plot(*lds_smoothed_xs_3D[idx].T, label=f'Trial {idx}', linewidth=2)
# # Add a darker segment for contrast, similar to the image
# ax.plot(x_data[:50], y_data[:50], z_data[:50], color='black', linewidth=3.5)


# --- Customization for axes and grid ---

# 1. Set the color of the panes (the "walls" of the 3D box)
pane_color_rgba = (1.0, 1.0, 1.0, 1.0) # White, Opaque
ax.xaxis.set_pane_color(pane_color_rgba)
ax.yaxis.set_pane_color(pane_color_rgba)
ax.zaxis.set_pane_color(pane_color_rgba)

# 2. Set the color and style of the pane edges (the lines forming the box)
pane_edge_color = 'white'
pane_edge_linewidth = 1.5 # Thicker edges for the box
ax.xaxis.pane.set_edgecolor(pane_edge_color)
ax.yaxis.pane.set_edgecolor(pane_edge_color)
ax.zaxis.pane.set_edgecolor(pane_edge_color)
ax.xaxis.pane.set_linewidth(pane_edge_linewidth)
ax.yaxis.pane.set_linewidth(pane_edge_linewidth)
ax.zaxis.pane.set_linewidth(pane_edge_linewidth)

# 3. Explicitly set tick locators to define where grid lines should be drawn
# This ensures there are ticks for the grid, even if we make them invisible later.
# Adjust the number of bins (e.g., 5-7) to control grid density.
num_grid_lines = 6
# ax.xaxis.set_major_locator(MaxNLocator(nbins=num_grid_lines))
# ax.yaxis.set_major_locator(MaxNLocator(nbins=num_grid_lines))
# ax.zaxis.set_major_locator(MaxNLocator(nbins=num_grid_lines))
ax.xaxis.set_major_locator(LinearLocator(numticks=num_grid_lines))
ax.yaxis.set_major_locator(LinearLocator(numticks=num_grid_lines))
ax.zaxis.set_major_locator(LinearLocator(numticks=num_grid_lines))

# 4. Configure the grid lines using _axinfo for robust styling
# This method directly modifies the properties Matplotlib uses for drawing.
grid_line_color = 'dimgray'
grid_line_width = 0.7 # Thinner than pane edges, adjust as needed

for axis_obj in [ax.xaxis, ax.yaxis, ax.zaxis]:
    axis_obj._axinfo["grid"].update({
        'color': grid_line_color,
        'linewidth': grid_line_width,
        'linestyle': '-'
    })
    # Ensure grid is on (though setting color/linewidth usually implies this)
    # If you had `ax.grid(False)` earlier, ensure this overrides it.
    # For _axinfo, setting the style IS enabling it.

# 5. Remove axis tick marks and labels visually
# (AFTER setting locators and styling grid, as grid relies on tick positions)

# Remove tick labels
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_zticklabels([])

# ax.set_xlabel('Neuron 1')
# ax.set_ylabel('Neuron 2')
# ax.set_zlabel('Neuron 3')

# Make tick marks invisible (zero length)
ax.tick_params(axis='x', length=0)
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='z', length=0)
# Alternatively, to remove tick marks completely (might also remove grid if not careful with locators):
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_zticks([])
# Using tick_params with length=0 is often safer when you want grids but no visible ticks.

# 6. (Optional) Set axis limits if needed, otherwise, Matplotlib will autoscale
# ax.set_xlim([-1.1, 1.1]) # Ensure data is within limits
# ax.set_ylim([-1.1, 1.1])
# ax.set_zlim([-1.1, 1.1])

# 7. (Optional) Set the view angle
ax.view_init(elev=20, azim=20) # Adjust elevation and azimuth
# plt.savefig("3D_LDS_dynamics_v2.png", bbox_inches='tight', facecolor='white', dpi=600)
plt.savefig('figure2_clds/LDS_dynamics.pdf', bbox_inches='tight', facecolor='white', transparent=True)
plt.show()

In [ ]:
subspace_angle_matrix = compute_subspace_angle_matrix_v2(Hs1[:, :, :])
inferred_normalized_grassmann_dist_matrix = jnp.linalg.norm(subspace_angle_matrix, ord=2, axis=-1) / (jnp.sqrt(state_dim) * jnp.pi / 2)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(5.5,2.5), gridspec_kw={'width_ratios': [1, 1.1], 'wspace': 0.1})

vmin = jnp.concatenate([normalized_grassmann_dist_matrix, inferred_normalized_grassmann_dist_matrix]).min()
vmax = jnp.concatenate([normalized_grassmann_dist_matrix, inferred_normalized_grassmann_dist_matrix]).max()

axs[0].imshow(normalized_grassmann_dist_matrix, aspect='auto', interpolation='none', vmin=vmin, vmax=vmax, cmap='inferno')
axs[0].set_ylabel('Trials')
axs[0].set_xlabel('Trials')
axs[0].set_yticks([1, 250, 500, 750])
axs[0].set_xticks([1, 250, 500, 750])
axs[0].set_title('Ground Truth Drift')

im = axs[1].imshow(inferred_normalized_grassmann_dist_matrix, aspect='auto', interpolation='none', vmin=vmin, vmax=vmax, cmap='inferno')
axs[1].set_yticks([])
axs[1].set_xticks([1, 250, 500, 750])
axs[1].set_xlabel('Trials')
axs[1].set_title('Learned Drift')
divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.1)
cbar = fig.colorbar(im, cax=cax)
cbar.ax.set_ylabel(r'Grassmann Distance $\in$ [0,1]')

plt.savefig('appendix_clds/figure2_toy_example_drift.pdf', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# # twist_angle_matrix = compute_twist_angle_matrix(true_emissions_weights[:, :, :])
# inferred_twist_angle_matrix = compute_twist_angle_matrix(torch.tensor(np.array(Hs1, copy=True), device='cuda'))
# inferred_twist_angle_matrix = jnp.array(inferred_twist_angle_matrix.cpu().numpy())
# d_in_max = jnp.pi * jnp.sqrt(true_state_dim // 2)
# inferred_normalized_twist_angle_matrix = jnp.clip(inferred_twist_angle_matrix / d_in_max, 0.0, 1.0)

In [ ]:
# fig, axs = plt.subplots(1, 2, figsize=(5.5,2.5), gridspec_kw={'width_ratios': [1, 1.1], 'wspace': 0.1})

# vmin = jnp.concatenate([normalized_twist_angle_matrix, inferred_normalized_twist_angle_matrix]).min()
# vmax = jnp.concatenate([normalized_twist_angle_matrix, inferred_normalized_twist_angle_matrix]).max()

# axs[0].imshow(normalized_twist_angle_matrix, aspect='auto', interpolation='none', vmin=vmin, vmax=vmax, cmap='inferno')
# axs[0].set_ylabel('Trials')
# axs[0].set_xlabel('Trials')
# axs[0].set_yticks([1, 250, 500, 750])
# axs[0].set_xticks([1, 250, 500, 750])
# axs[0].set_title('Ground Truth Drift')

# im = axs[1].imshow(inferred_normalized_twist_angle_matrix, aspect='auto', interpolation='none', vmin=vmin, vmax=vmax, cmap='inferno')
# axs[1].set_yticks([])
# axs[1].set_xticks([1, 250, 500, 750])
# axs[1].set_xlabel('Trials')
# axs[1].set_title('Learned Drift')
# divider = make_axes_locatable(axs[1])
# cax = divider.append_axes("right", size="5%", pad=0.1)
# cbar = fig.colorbar(im, cax=cax)
# cbar.ax.set_ylabel(r'Within Subspace Rotation $\in$ [0,1]')


# # plt.savefig('figure2_clds/figure2_toy_example_drift.pdf', bbox_inches='tight', facecolor='white')
# plt.show()

In [ ]:
smds_rotation, explained_variances = compute_rotation(emissions, Hs1)
rotated_inferred_emissions_weights = jnp.einsum('rij,jk->rik', Hs1, smds_rotation)
inferred_evs = vmap(compute_explained_variance, in_axes=(None, None, -1))(emissions, rotated_inferred_emissions_weights, 
                                                                          rotated_inferred_emissions_weights[:, :, None])

A = smds_models[0][1].dynamics.weights
smds_rotation_inv = jnp.linalg.inv(smds_rotation)
transformed_smds_A = smds_rotation @ A @ smds_rotation_inv

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(D), inferred_evs)
ax.set_ylabel('explained variance')
ax.set_xlabel('state dim')
plt.show()

In [ ]:
rotated_inferred_cos_sims_diag = np.zeros((D, num_trials-1))
rotated_inferred_cos_dists_diag = np.zeros((D, num_trials-1))
for d in tqdm(range(D)):
    rotated_inferred_cos_sims_diag[d] = vmap(compute_cos_sim, in_axes=(0, 0))(rotated_inferred_emissions_weights[1:,:,d], 
                                                                          rotated_inferred_emissions_weights[:-1,:,d])
    rotated_inferred_cos_dists_diag[d] = vmap(compute_cos_dist, in_axes=(0, 0))(rotated_inferred_emissions_weights[1:,:,d], 
                                                                          rotated_inferred_emissions_weights[:-1,:,d])

inferred_diag_thetas = np.rad2deg(jnp.arccos(jnp.clip(rotated_inferred_cos_sims_diag, min=-1, max=1)))
inferred_diag_thetas = np.array(inferred_diag_thetas)

rotated_inferred_cos_sims = np.zeros((D, num_trials))
rotated_inferred_cos_dists = np.zeros((D, num_trials))
for d in tqdm(range(D)):
    rotated_inferred_cos_sims[d] = vmap(compute_cos_sim, in_axes=(0, None))(rotated_inferred_emissions_weights[:,:,d], 
                                                                          rotated_inferred_emissions_weights[0,:,d])
    rotated_inferred_cos_dists[d] = vmap(compute_cos_dist, in_axes=(0, None))(rotated_inferred_emissions_weights[:,:,d], 
                                                                          rotated_inferred_emissions_weights[0,:,d])

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
# fig, ax = plt.subplots(figsize=(8,4))
plt.grid(True, zorder=0)
ax.scatter(true_evs, true_diag_thetas.mean(1), s=50, color='k', label='True SMDS', zorder=3)
ax.scatter(inferred_evs, inferred_diag_thetas.mean(1), s=50, color='r', label='Learned SMDS', zorder=3)
ax.set_title('Drift Rate vs. Explained Variance Per Axis')

ax.set_xlabel('Explained variance')
# ax.set_ylabel('mean(' + r'$|\Delta\theta|$' + ' per trial) in degrees')
ax.set_ylabel('average ' + r'$|\Delta\theta|$' + ' per trial\nin degrees')
ax.legend()

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# plt.ylim(0, 1.0)

# plt.savefig('figure2_clds/figure2_drift_rate_final.pdf', bbox_inches='tight', facecolor='white')
plt.show()

plt.plot(rotated_true_cos_dists.T, color='k')
plt.plot(rotated_inferred_cos_dists.T, color='r')
plt.ylabel('cosine distance')
plt.xlabel('trial')
plt.show()

In [ ]:
rotated_inferred_cos_sims = np.zeros((D, num_trials, num_trials))
rotated_inferred_cos_dists = np.zeros((D, num_trials, num_trials))
for d in tqdm(range(D)):
    for b in range(num_trials):
        rotated_inferred_cos_sims[d, b] = vmap(compute_cos_sim, in_axes=(None, 0))(rotated_inferred_emissions_weights[b,:,d], 
                                                                              rotated_inferred_emissions_weights[:,:,d])
        rotated_inferred_cos_dists[d, b] = vmap(compute_cos_dist, in_axes=(None, 0))(rotated_inferred_emissions_weights[b,:,d], 
                                                                              rotated_inferred_emissions_weights[:,:,d])

In [ ]:
inferred_thetas = np.rad2deg(jnp.arccos(jnp.clip(rotated_inferred_cos_sims, min=-1, max=1)))
inferred_thetas = np.array(inferred_thetas)

In [ ]:
max_drift_amounts = []
for i in range(D):
    max_drift_amounts.append(inferred_thetas[i][jnp.triu_indices(inferred_thetas.shape[-1], k=1)].max())

In [ ]:
rotated_true_cos_sims = np.zeros((D, num_trials, num_trials))
rotated_true_cos_dists = np.zeros((D, num_trials, num_trials))
for d in tqdm(range(D)):
    for b in range(num_trials):
        rotated_true_cos_sims[d, b] = vmap(compute_cos_sim, in_axes=(None, 0))(rotated_true_emissions_weights[b,:,d], 
                                                                              rotated_true_emissions_weights[:,:,d])
        rotated_true_cos_dists[d, b] = vmap(compute_cos_dist, in_axes=(None, 0))(rotated_true_emissions_weights[b,:,d], 
                                                                              rotated_true_emissions_weights[:,:,d])

In [ ]:
true_thetas = np.rad2deg(jnp.arccos(jnp.clip(rotated_true_cos_sims, min=-1, max=1)))
true_thetas = np.array(true_thetas)

In [ ]:
true_max_drift_amounts = []
for i in range(D):
    true_max_drift_amounts.append(true_thetas[i][jnp.triu_indices(true_thetas.shape[-1], k=1)].max())

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
plt.grid(True, zorder=0)
ax.scatter(true_evs, true_max_drift_amounts, s=50, color='k', label='True SMDS', zorder=3)
ax.scatter(inferred_evs, max_drift_amounts, s=50, color='r', label='Learned SMDS', zorder=3)
# ax.set_title('Drift Rate vs. Explained Variance Per Axis')

# ax.set_xlabel('Explained variance')
# ax.set_ylabel('mean(' + r'$|\Delta\theta|$' + ' per trial) in degrees')
# ax.set_ylabel('average ' + r'$|\Delta\theta|$' + ' per trial\nin degrees')
ax.legend()

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.ylim(0, 110)
# plt.xlim(0, 0.2)

# plt.savefig('figure2_clds/figure2_max_drift.pdf', bbox_inches='tight', 
#             facecolor='white', transparent=True)
plt.show()